# Project 4

## DELIVRABLE A

In [1]:
!pip install CFEDemands eep153_tools -q

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cfe.regression as rgsn

In [27]:
exp19 = pd.read_csv("Tanzania - Food Expenditures (2019-20).csv")
exp20 = pd.read_csv("Tanzania - Food Expenditures (2020-21).csv")

prices19 = pd.read_csv("Tanzania - Food Prices (2019-20).csv")
prices20 = pd.read_csv("Tanzania - Food Prices (2020-21).csv")

hh = pd.read_csv("Tanzania - Household Characteristics.csv")
fct = pd.read_csv("Tanzania - FCT.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'Tanzania - Food Expenditures (2019-20).csv'

In [28]:
expenditures = pd.concat([exp19, exp20], ignore_index=True)
prices = pd.concat([prices19, prices20], ignore_index=True)

expenditures["Expenditure"] = pd.to_numeric(expenditures["Expenditure"], errors="coerce")
prices["Price"] = pd.to_numeric(prices["Price"], errors="coerce")

expenditures = expenditures.dropna(subset=["Expenditure"])
prices = prices.dropna(subset=["Price"])

prices = prices.replace(0, np.nan)

NameError: name 'exp19' is not defined

In [29]:
x = expenditures.set_index(["i", "t", "m", "j"])["Expenditure"]

z = hh.set_index(["i", "t", "m"])
z = z.apply(pd.to_numeric, errors="coerce")

common_index = x.index.droplevel("j").intersection(z.index)
x = x[x.index.droplevel("j").isin(common_index)]
z = z.loc[common_index].drop_duplicates()

NameError: name 'expenditures' is not defined

In [30]:
r = rgsn.Regression(y=np.log(x), d=z)

r.to_pickle("tanzania_demand_system.pickle")

NameError: name 'x' is not defined

In [31]:
r = rgsn.read_pickle("tanzania_demand_system.pickle")

FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/EEP153_Materials/Project4/tanzania_demand_system.pickle'

In [32]:
p = prices[prices["u"] == "kg"].copy()

p_wide = p.pivot_table(
index=["t", "m"],
columns="j",
values="Price",
aggfunc="mean"
)

p_wide.columns.name = "j"

pbar = p_wide.mean()

pbar = pbar[r.beta.index]

pbar.head()

NameError: name 'prices' is not defined

In [33]:
xhat = r.predicted_expenditures()

xbar = xhat.groupby(["i", "t", "m"]).sum()

xref = xbar.quantile(0.5)

print("Median household food budget:", xref)

NameError: name 'r' is not defined

In [34]:
baseline_demands = r.demands(xref, pbar)

baseline_demands.sort_values(ascending=False).head(15)

NameError: name 'r' is not defined

In [35]:
def change_price(food, multiplier, p=pbar):
    new_p = p.copy()
    new_p.loc[food] = new_p.loc[food] * multiplier
    return new_p

NameError: name 'pbar' is not defined

In [36]:
food = "Maize"

new_p = change_price(food, 1.5)

new_demands = r.demands(xref, new_p)

comparison = pd.DataFrame({
"Baseline Demand": baseline_demands,
"After Price Increase": new_demands,
"Change": new_demands - baseline_demands,
"Percent Change": ((new_demands - baseline_demands) / baseline_demands) * 100
})

comparison.loc[food]

NameError: name 'change_price' is not defined

In [37]:
food = "Maize"

scale = np.linspace(0.5, 2, 20)

median_budget = xbar.quantile(0.5)
low_budget = xbar.quantile(0.25)
high_budget = xbar.quantile(0.75)

plt.plot(
    [r.demands(median_budget, change_price(food, s))[food] for s in scale],
    scale,
    label="Median budget"
)

plt.plot(
    [r.demands(low_budget, change_price(food, s))[food] for s in scale],
    scale,
    label="Low budget"
)

plt.plot(
    [r.demands(high_budget, change_price(food, s))[food] for s in scale],
    scale,
    label="High budget"
)

plt.xlabel(f"Quantity of {food} demanded")
plt.ylabel("Price relative to baseline")
plt.title(f"Demand Curve for {food} in Tanzania")
plt.legend()
plt.show()

NameError: name 'xbar' is not defined

In [38]:

cash_transfer_demands = r.demands(1.2 * xref, pbar)

cash_transfer_comparison = pd.DataFrame({
    "Baseline Demand": baseline_demands,
    "After 20 Percent Budget Increase": cash_transfer_demands,
    "Change": cash_transfer_demands - baseline_demands,
    "Percent Change": ((cash_transfer_demands - baseline_demands) / baseline_demands) * 100
})

cash_transfer_comparison.sort_values("Percent Change", ascending=False).head(15)

NameError: name 'r' is not defined

In [39]:
food = "Eggs"

scale = np.geomspace(0.1, 10, 50)

shares = [
    (r.expenditures(s * xref, pbar) / (s * xref))[food]
    for s in scale
]

plt.plot(np.log(scale), shares)
plt.xlabel(f"Log budget relative to baseline budget of {xref:.0f}")
plt.ylabel(f"Expenditure share for {food}")
plt.title(f"Engel Curve for {food} in Tanzania")
plt.show()

NameError: name 'r' is not defined

In [40]:
fct_clean = fct.set_index("j")
fct_clean = fct_clean.apply(pd.to_numeric, errors="coerce")


NameError: name 'fct' is not defined

In [41]:
common_foods = baseline_demands.index.intersection(fct_clean.index)

baseline_nutrients = fct_clean.loc[common_foods].T @ baseline_demands.loc[common_foods]

baseline_nutrients.sort_values(ascending=False)


NameError: name 'baseline_demands' is not defined

In [42]:

common_foods = cash_transfer_demands.index.intersection(fct_clean.index)

cash_transfer_nutrients = fct_clean.loc[common_foods].T @ cash_transfer_demands.loc[common_foods]

nutrient_comparison = pd.DataFrame({
    "Baseline": baseline_nutrients,
    "After Cash Transfer": cash_transfer_nutrients,
    "Change": cash_transfer_nutrients - baseline_nutrients,
    "Percent Change": ((cash_transfer_nutrients - baseline_nutrients) / baseline_nutrients) * 100
})

nutrient_comparison

NameError: name 'cash_transfer_demands' is not defined

In [43]:
policy_summary = pd.DataFrame({
    "Policy Scenario": [
        "50 percent increase in maize price",
        "20 percent increase in household food budget"
    ],
    "Expected Effect": [
        "Lower demand for maize and possible substitution toward other foods",
        "Higher food demand and possible improvement in nutrient intake"
    ],
    "Policy Meaning": [
        "Food price increases can hurt household food security, especially for lower-income households",
        "Cash transfers can improve access to food and support better nutrition outcomes"
    ]
})

policy_summary


,Policy Scenario,Expected Effect,Policy Meaning
0,50 percent increase in maize price,Lower demand for maize and possible substituti...,Food price increases can hurt household food s...
1,20 percent increase in household food budget,Higher food demand and possible improvement in...,Cash transfers can improve access to food and ...
